# Capstone build --- Chapter 5: Tools as Typed Actions

The loop in Chapter~1 had an action space of one tool. Chapter~5 makes the action space the full set the complaint agent draws from: five typed tools, each a named action over a Pydantic input and output model. This notebook widens the registry to all five and shows the property that makes the action space safe --- a malformed proposal is rejected against the input schema at the registry boundary, before any tool body runs.

## The full typed action space

`register_all` binds the five tools into a registry. Four are module-level tool objects; the policy-search tool is constructed against the policy directory because its retrieval reads the deployed store. The path is resolved so the notebook runs whether the working directory is the notebooks folder or the project root.

In [ ]:
from agentlab.tools import ToolRegistry
from agentlab.capstone import banking_tools
from pathlib import Path

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'policies').exists()), Path('.'))
registry = ToolRegistry()
banking_tools.register_all(registry, policies_dir=root / 'data' / 'policies')
for tool in registry.all():
    print(f'{tool.name:20s} {tool.input_schema.__name__:14s} '
          f'-> {tool.output_schema.__name__:14s} risk={tool.risk.value}')

The registry is now the agent's action space: the set of typed actions it may propose. Each entry pairs an input contract with an output contract, so the harness can check a call on the way in and a result on the way out. The risk column records how much scrutiny each action warrants, which the gate stack in Chapter~6 reads when it decides whether to authorize a call.

## A malformed call is rejected at the boundary

The registry validates a proposed call against the tool's input schema before the tool runs. A well-formed proposal is parsed into the typed input model; a proposal missing a required field raises a validation error, so an ill-formed action never reaches the tool body. This is the same `registry.validate` the environment called in Chapter~1, seen now as the boundary check it is.

In [ ]:
from pydantic import ValidationError

ok = registry.validate(
    'classify_complaint',
    {'message': 'I was charged a $35 overdraft fee I did not authorize.'},
)
print('accepted   :', type(ok).__name__, '->', ok)

try:
    registry.validate('classify_complaint', {})   # missing the message field
except ValidationError as e:
    err = e.errors()[0]
    print('rejected    :', err['type'], '-', err['loc'])

This is the capstone's realization of Chapter~5: the agent's action space is a registry of typed tools, and every proposal is validated against a declared schema before it can act. The loop from Chapter~1 now proposes from these five actions instead of one. Chapter~6 wraps the registry in the governed executor that adds the gate stack around this boundary check, and Chapter~15 assembles the five tools and the gates into the shipped harness, at which point the action space here is exactly the one `build_complaint_harness` registers.